# Quickstart on synthetic data

This notebook runs the full `transplant_rates` pipeline on the bundled **synthetic**
dataset. Every value in that dataset is randomly generated, with no real SRTR/OASIM
data and no cluster paths, so the notebook can be run anywhere after cloning the
repository.

If the synthetic files are missing, the setup cell regenerates them.

## Setup

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

# Locate the examples/ folder whether the notebook is run from the repo root
# or from inside examples/.
cwd = Path.cwd()
examples_dir = cwd if cwd.name == "examples" else cwd / "examples"
data_dir = examples_dir / "synthetic_data"

# Make `import transplant_rates` work from a source checkout (harmless if the
# package is pip-installed), and make the data generator importable.
sys.path.insert(0, str(examples_dir))
sys.path.insert(0, str(examples_dir.parent.parent))

# Generate the synthetic dataset if it is not already present.
if not (data_dir / "cleaned_static_candidate_data.csv").exists():
    import generate_synthetic_data as gen
    gen.main()

from transplant_rates import TransplantRatesCalculator, OffersCalculator

SIM_BEGIN = datetime(2021, 3, 15)
SIM_END = datetime(2022, 3, 15)

core_paths = dict(
    donor_path=data_dir / "cleaned_donor_data.csv",
    candidate_path=data_dir / "cleaned_timevarying_KAS_candidate_data.csv",
    static_path=data_dir / "cleaned_static_candidate_data.csv",
    timevarying_path=data_dir / "cleaned_timevarying_candidate_data.csv",
    gen_removal_path=data_dir / "generated_removal_candidate_data.csv",
    gen_hist_path=data_dir / "generated_history_candidate_data.csv",
    race_path=data_dir / "can_race.csv",
)

## 1. Load core data

`load_core_data` reads the OASIM-style candidate/donor tables and the SRTR race file,
filters to kidney candidates, and computes each candidate's age, EPTS, and CPRA at
the simulation start.

In [ ]:
calc = TransplantRatesCalculator(SIM_BEGIN, SIM_END, cache_dir=None)
calc.load_core_data(**core_paths)

print(f"kidney candidates: {len(calc.static)}")
print(f"donors:            {len(calc.donor)}")

## 2. Compare policies (summary table)

Register a single policy ("Current") plus a two-iteration policy group ("Proposed").
For the group, the summary reports the mean / min / max transplant rate across
iterations. Rates are transplants per patient-year, stratified by subgroup.

In [ ]:
current = data_dir / "policy_current.txt"
calc.add_policies([current], name_map={current: "Current"})
calc.add_policy_group(
    [data_dir / "policy_proposed_iter1.txt", data_dir / "policy_proposed_iter2.txt"],
    group_name="Proposed",
)

summary = calc.compute_summary_table()
summary

## 3. Subgroup transplant rates

Convenience accessors return the per-subgroup rates for a single policy.

In [ ]:
age_rates = calc.get_age_rates(current)
race_rates = calc.get_race_rates(current)

print("Age-specific rates (per patient-year):")
for (lo, hi), rate in age_rates.items():
    print(f"  {lo:>2}-{hi:<2}: {rate:.3f}")

print("\nRace-specific rates (per patient-year):")
for race, rate in race_rates.items():
    print(f"  {race:<8}: {rate:.3f}")

## 4. Visualization: EPTS x KDPI heatmap

Distribution of transplants across recipient EPTS bins and donor KDPI bins for the
current policy.

In [ ]:
calc.plot_kdpi_heatmap(current, step=10)

## 5. Offers analysis

`OffersCalculator` works on offer-level files (one row per candidate-kidney offer).
It reuses the same core data for EPTS / KDPI / race joins.

In [ ]:
oc = OffersCalculator(SIM_BEGIN, SIM_END, cache_dir=None)
oc.load_core_data(**core_paths)

offers = data_dir / "offers_current.txt"
oc.register_offers([offers], name_map={offers: "Current"})

print("Offers per kidney:")
for k, v in oc.summarize_offers_per_kidney(offers).items():
    print(f"  {k}: {v}")

In [ ]:
oc.summarize_kdpi_by_race(offers)

In [ ]:
oc.plot_epts_kdpi_catplot([offers], epts_step=0.2)

## Next steps

For a real analysis, point `load_core_data(...)` at your own OASIM-generated
candidate/donor files and SRTR `can_race.csv`, and pass your simulator's policy
output `.txt` files to `add_policies(...)` / `add_policy_group(...)`. See the
top-level `README.md` for the expected schema.